In [51]:
import os
import wandb
import torch
from pathlib import Path
from module.utils.data_module import fMRIDataModule
from module.pl_classifier import LitClassifier
import pytorch_lightning as pl

# wandb API 객체 생성
api = wandb.Api()

# 프로젝트명과 실험 ID를 통해 실험(run) 가져오기
run_id = "rqvmz6lo"  # 실제 run_id
run = api.run("snu-connectome/moviefmri/runs/" + run_id)

# 해당 실험의 config 가져오기
config = run.config

# wandb artifact 다운받기 (이미 있으면 다시 다운받지 않도록 처리)
artifact_name = "best_model:v18"
artifact_dir = Path(f"/data/scratch/kimbo/SwiFT-IO/src/artifacts/{artifact_name}")

if not artifact_dir.exists():
    artifact = api.artifact(f"snu-connectome/moviefmri/{artifact_name}", type="model")
    artifact_dir = Path(artifact.download())  # 다운로드
    print(f"Artifact downloaded to: {artifact_dir}")
else:
    print(f"Artifact already exists at: {artifact_dir}")

# 체크포인트 파일 자동으로 선택 (artifact_dir에서 .ckpt로 끝나는 파일 찾기)
ckpt_files = list(artifact_dir.glob("checkpt*.ckpt"))

if ckpt_files:
    # 가장 최신 체크포인트 파일 선택 (수정된 날짜 기준)
    checkpoint_path = sorted(ckpt_files, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    print(f"Selected checkpoint: {checkpoint_path}")
else:
    raise FileNotFoundError("No checkpoint file found.")

# 데이터 모듈과 하이퍼파라미터 설정
# config에서 하이퍼파라미터를 바로 가져오기
data_module = fMRIDataModule(
    batch_size=config['batch_size'], 
    num_workers=config['num_workers'], 
    image_path=config['image_path'],
    pretraining = config['pretraining'],
    dataset_name = config['dataset_name'],
    dataset_split_num = config['dataset_split_num'],
    use_contrastive = config['use_contrastive'],
    contrastive_type = config['contrastive_type'],
    stride_between_seq = config['stride_between_seq'],
    stride_within_seq = config['stride_within_seq'],
    with_voxel_norm = config['with_voxel_norm'],
    downstream_task = config['downstream_task'],
    downstream_task_type = config['downstream_task_type'],
    shuffle_time_sequence = config['shuffle_time_sequence'],
    input_type = config['input_type'],
    decoder = config['decoder'],
    adjust_hrf = config['adjust_hrf'],
    strategy = config['strategy'],
    limit_training_samples = config['limit_training_samples'],
    label_scaling_method=config.get('label_scaling_method', 'standardization'),  # 기본값 추가
    bad_subj_path=config.get('bad_subj_path', None),  # 기본값 처리
    sequence_length=config['sequence_length'],  # 예시로 추가된 하이퍼파라미터
    eval_batch_size=config['eval_batch_size'],  # 예시로 추가된 하이퍼파라미터
    train_split=config['train_split'],  # 예시로 추가된 하이퍼파라미터
    val_split=config['val_split']  # 예시로 추가된 하이퍼파라미터
)

# 데이터셋 준비
data_module.setup()
data_module.prepare_data()

# 모델 초기화
model = LitClassifier(data_module=data_module, **config)

# 체크포인트 로드
model.load_state_dict(torch.load(checkpoint_path, map_location='cpu')['state_dict'])
model.to('cpu')
model.eval()

# trainer 설정 (CPU에서 실행)
trainer = pl.Trainer(gpus=0)

# 모델 테스트
test_results = trainer.test(model, datamodule=data_module)

# 테스트 결과 출력
print(f"Test results: {test_results}")


Artifact already exists at: /data/scratch/kimbo/SwiFT-IO/src/artifacts/best_model:v18
Selected checkpoint: /data/scratch/kimbo/SwiFT-IO/src/artifacts/best_model:v18/checkpt-epoch=09-valid_mse=0.02.ckpt
Number of sequences: 11809
Number of unique subjects: 473
Number of sequences: 2525
Number of unique subjects: 101
Number of sequences: 2528
Number of unique subjects: 103
number of train_subj: 473
number of val_subj: 101
number of test_subj: 103
length of train_idx: 11809
length of val_idx: 2525
length of test_idx: 2528
Number of sequences: 11809
Number of unique subjects: 473
Number of sequences: 2525
Number of unique subjects: 101
Number of sequences: 2528
Number of unique subjects: 103
number of train_subj: 473
number of val_subj: 101
number of test_subj: 103
length of train_idx: 11809
length of val_idx: 2525
length of test_idx: 2528
target_mean:0.7645551213965138, target_std:2.0855093618420533
swin4d_ver7


/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:466: LightningDeprecationWarning: Setting `Trainer(gpus=0)` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=0)` instead.
  rank_zero_deprecation(
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:166: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data/scratch/kimbo/.conda/swiftio/lib/python3.9/sit ...
  rank_zero_warn(
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/pytorch_lightning/trainer/setup.py:176: PossibleUserWarning: GPU availabl

Number of sequences: 11809
Number of unique subjects: 473
Number of sequences: 2525
Number of unique subjects: 101


Missing logger folder: /data/scratch/kimbo/SwiFT-IO/lightning_logs


Number of sequences: 2528
Number of unique subjects: 103
number of train_subj: 473
number of val_subj: 101
number of test_subj: 103
length of train_idx: 11809
length of val_idx: 2525
length of test_idx: 2528
Testing: 0it [00:00, ?it/s]

/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedS

Testing DataLoader 0:   0%|          | 0/632 [00:00<?, ?it/s]

RuntimeError: mat1 and mat2 must have the same dtype

In [64]:
import os
import wandb
import torch
from pathlib import Path
from module.utils.data_module import fMRIDataModule
from module.pl_classifier import LitClassifier
import pytorch_lightning as pl

# wandb API 객체 생성
api = wandb.Api()

# 프로젝트명과 실험 ID를 통해 실험(run) 가져오기
run_id = "rqvmz6lo"  # 실제 run_id
run = api.run("snu-connectome/moviefmri/runs/" + run_id)

# 해당 실험의 config 가져오기
config = run.config

# wandb artifact 다운받기 (이미 있으면 다시 다운받지 않도록 처리)
artifact_name = "best_model:v18"
artifact_dir = Path(f"/data/scratch/kimbo/SwiFT-IO/src/artifacts/{artifact_name}")

if not artifact_dir.exists():
    artifact = api.artifact(f"snu-connectome/moviefmri/{artifact_name}", type="model")
    artifact_dir = Path(artifact.download())  # 다운로드
    print(f"Artifact downloaded to: {artifact_dir}")
else:
    print(f"Artifact already exists at: {artifact_dir}")

# 체크포인트 파일 자동으로 선택 (artifact_dir에서 .ckpt로 끝나는 파일 찾기)
ckpt_files = list(artifact_dir.glob("checkpt*.ckpt"))

if ckpt_files:
    # 가장 최신 체크포인트 파일 선택 (수정된 날짜 기준)
    checkpoint_path = sorted(ckpt_files, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    print(f"Selected checkpoint: {checkpoint_path}")
else:
    raise FileNotFoundError("No checkpoint file found.")

# 데이터 모듈과 하이퍼파라미터 설정
# config에서 하이퍼파라미터를 바로 가져오기
data_module = fMRIDataModule(
    batch_size=config['batch_size'], 
    num_workers=config['num_workers'], 
    image_path=config['image_path'],
    pretraining = config['pretraining'],
    dataset_name = config['dataset_name'],
    dataset_split_num = config['dataset_split_num'],
    use_contrastive = config['use_contrastive'],
    contrastive_type = config['contrastive_type'],
    stride_between_seq = config['stride_between_seq'],
    stride_within_seq = config['stride_within_seq'],
    with_voxel_norm = config['with_voxel_norm'],
    downstream_task = config['downstream_task'],
    downstream_task_type = config['downstream_task_type'],
    shuffle_time_sequence = config['shuffle_time_sequence'],
    input_type = config['input_type'],
    decoder = config['decoder'],
    adjust_hrf = config['adjust_hrf'],
    strategy = config['strategy'],
    limit_training_samples = config['limit_training_samples'],
    label_scaling_method=config.get('label_scaling_method', 'standardization'),  # 기본값 추가
    bad_subj_path=config.get('bad_subj_path', None),  # 기본값 처리
    sequence_length=config['sequence_length'],  # 예시로 추가된 하이퍼파라미터
    eval_batch_size=config['eval_batch_size'],  # 예시로 추가된 하이퍼파라미터
    train_split=config['train_split'],  # 예시로 추가된 하이퍼파라미터
    val_split=config['val_split']  # 예시로 추가된 하이퍼파라미터
)

# 데이터셋 준비
data_module.setup()
data_module.prepare_data()

# 모델 초기화
model = LitClassifier(data_module=data_module, **config)
# model = model.half().to('cuda')

# 체크포인트 로드
checkpoint = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['state_dict'])
model.to('cpu')
model.eval()

# 데이터 타입 확인 및 변환을 위해 forward() 함수 수정
class PatchEmbed(nn.Module):
    def forward(self, x):
        # 데이터 타입이 일치하는지 확인
        if x.dtype != torch.float32:
            x = x.float()  # 입력 데이터 타입을 float32로 변환

        assert x.shape[2] == self.img_size[1], f"Input image width ({x.shape[2]}) doesn't match model ({self.img_size[1]})."
        assert x.shape[3] == self.img_size[2], f"Input image width ({x.shape[3]}) doesn't match model ({self.img_size[2]})."

        # 패치 임베딩 수행
        x = self.proj(x)
        return x

# 테스트할 때도 동일한 변환을 적용
def check_input_and_weight(model, batch):
    # 모델의 weight 데이터 타입 확인
    for name, param in model.named_parameters():
        print(f"Weight dtype of {name}: {param.dtype}")
    
    # 배치의 전체 내용을 출력하여 확인
    print(f"Batch keys: {batch.keys()}")  # batch가 dict라면 keys()로 확인
    
    # 입력 데이터 타입 확인
    if isinstance(batch, dict):
        fmri = batch.get('fmri_sequence', None)  # 데이터에서 fmri를 찾는 예시
    else:
        fmri = batch
    
    if fmri is not None:
        print(f"Input dtype: {fmri.dtype}")
        fmri = fmri.float()  # 입력 데이터 타입을 float32로 변환
        print(f"Converted input dtype: {fmri.dtype}")
    else: 
        print("No input data found in batch")

# 위와 같은 변경을 한 후 테스트 코드 실행
for batch in data_module.test_dataloader():
    check_input_and_weight(model, batch)
    break  # 첫 번째 배치만 확인

# trainer 설정 (CPU에서 실행)
trainer = pl.Trainer(gpus=0)
# trainer = pl.Trainer(accelerator='gpu', devices=1, precision=16)


# 모델 테스트
test_results = trainer.test(model, datamodule=data_module)

# 테스트 결과 출력
print(f"Test results: {test_results}")


Artifact already exists at: /data/scratch/kimbo/SwiFT-IO/src/artifacts/best_model:v18
Selected checkpoint: /data/scratch/kimbo/SwiFT-IO/src/artifacts/best_model:v18/checkpt-epoch=09-valid_mse=0.02.ckpt
Number of sequences: 11809
Number of unique subjects: 473
Number of sequences: 2525
Number of unique subjects: 101
Number of sequences: 2528
Number of unique subjects: 103
number of train_subj: 473
number of val_subj: 101
number of test_subj: 103
length of train_idx: 11809
length of val_idx: 2525
length of test_idx: 2528
Number of sequences: 11809
Number of unique subjects: 473
Number of sequences: 2525
Number of unique subjects: 101
Number of sequences: 2528
Number of unique subjects: 103
number of train_subj: 473
number of val_subj: 101
number of test_subj: 103
length of train_idx: 11809
length of val_idx: 2525
length of test_idx: 2528
target_mean:0.7645551213965138, target_std:2.0855093618420533
swin4d_ver7


/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedS

Weight dtype of model.patch_embed.fc.weight: torch.float32
Weight dtype of model.patch_embed.fc.bias: torch.float32
Weight dtype of model.pos_embeds.0.pos_embed: torch.float32
Weight dtype of model.pos_embeds.0.time_embed: torch.float32
Weight dtype of model.pos_embeds.1.pos_embed: torch.float32
Weight dtype of model.pos_embeds.1.time_embed: torch.float32
Weight dtype of model.pos_embeds.2.pos_embed: torch.float32
Weight dtype of model.pos_embeds.2.time_embed: torch.float32
Weight dtype of model.pos_embeds.3.pos_embed: torch.float32
Weight dtype of model.pos_embeds.3.time_embed: torch.float32
Weight dtype of model.layers.0.blocks.0.norm1.weight: torch.float32
Weight dtype of model.layers.0.blocks.0.norm1.bias: torch.float32
Weight dtype of model.layers.0.blocks.0.attn.qkv.weight: torch.float32
Weight dtype of model.layers.0.blocks.0.attn.qkv.bias: torch.float32
Weight dtype of model.layers.0.blocks.0.attn.proj.weight: torch.float32
Weight dtype of model.layers.0.blocks.0.attn.proj.bias

/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:466: LightningDeprecationWarning: Setting `Trainer(gpus=0)` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=0)` instead.
  rank_zero_deprecation(
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:166: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data/scratch/kimbo/.conda/swiftio/lib/python3.9/sit ...
  rank_zero_warn(
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/pytorch_lightning/trainer/setup.py:176: PossibleUserWarning: GPU availabl

Number of sequences: 11809
Number of unique subjects: 473
Number of sequences: 2525
Number of unique subjects: 101
Number of sequences: 2528
Number of unique subjects: 103
number of train_subj: 473
number of val_subj: 101
number of test_subj: 103
length of train_idx: 11809
length of val_idx: 2525
length of test_idx: 2528
Testing: 0it [00:00, ?it/s]

/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/data/scratch/kimbo/.conda/swiftio/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedS

Testing DataLoader 0:   0%|          | 0/632 [00:00<?, ?it/s]

RuntimeError: mat1 and mat2 must have the same dtype

In [62]:
test_dataloader = data_module.test_dataloader()
test_dataloader